# 1D Burgers' Equation - Non-Linear Convection and Diffusion

This notebook explores the **Burgers' Equation**, detailing the **Cole-Hopf Transformation** which maps the non-linear problem to a linear heat equation.

## 1. Governing Equation

The 1D viscous Burgers' equation is:
$$\frac{\partial u}{\partial t} + u \frac{\partial u}{\partial x} = \nu \frac{\partial^2 u}{\partial x^2}$$

Where:
- $u(x, t)$: Flow velocity
- $\nu$: Kinematic viscosity

## 2. Detailed Analytical Trace (Cole-Hopf)

The Cole-Hopf transformation defined as $u = -2\nu \frac{1}{\phi} \frac{\partial \phi}{\partial x}$ linearizes the equation.

In [ ]:
import sympy as sp
from IPython.display import display, Math

x, t, nu = sp.symbols('x t nu', real=True, positive=True)
phi = sp.Function('phi')(x, t)

# 1. Define the Transformation: u = -2 * nu * (phi_x / phi)
u = -2 * nu * phi.diff(x) / phi

# 2. Substitute into Burgers' Equation members
u_t = u.diff(t)
u_x = u.diff(x)
u_xx = u.diff(x, x)

burgers_lhs = u_t + u * u_x
burgers_rhs = nu * u_xx

equation_residual = sp.simplify(burgers_lhs - burgers_rhs)

display(Math(f"\\text{{Residual after transformation: }} {sp.latex(equation_residual)} = 0"))

display(Math("\\text{The residual vanishes if } \phi \text{ satisfies the Heat Equation: } \\frac{\\partial \\phi}{\\partial t} = \\nu \\frac{\\partial^2 \\phi}{\\partial x^2}"))

## 3. Numerical Verification

We compare the non-linear numerical update against the theoretical behavior (wave steepening and diffusion).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def solve_burgers_numpy(nx, L, nt, dt, nu, u_init):
    dx = L / (nx - 1)
    u = u_init.copy()
    for _ in range(nt):
        un = u.copy()
        for i in range(1, nx - 1):
            # Upwind scheme for non-linear convection
            if un[i] >= 0:
                conv = un[i] * (un[i] - un[i-1]) / dx
            else:
                conv = un[i] * (un[i+1] - un[i]) / dx
            diff = nu * (un[i+1] - 2*un[i] + un[i-1]) / dx**2
            u[i] = un[i] + dt * (diff - conv)
    return u

nx = 101
L = 2.0
nu_val = 0.05
nt = 100
dt = 0.001
x_vals = np.linspace(0, L, nx)

u0 = np.exp(-100 * (x_vals - 0.5)**2)
u_final = solve_burgers_numpy(nx, L, nt, dt, nu_val, u0)

plt.figure(figsize=(10, 5))
plt.plot(x_vals, u0, 'k:', label='Initial condition')
plt.plot(x_vals, u_final, 'b-', label='Numerical Result')
plt.title("Burgers Equation: Non-linear Pulse Evolution")
plt.legend()
plt.grid(True)
plt.show()